# Mini-Projeto 3 — FitTech Knowledge Graph

**Autores:** Flávio Mesquita Marinho Filho e John Victor de Oliveira Atanazio

Este notebook transforma o domínio dos Mini-Projetos anteriores em uma base RDF/RDFS/OWL. Ele carrega `fittech.ttl`, valida os requisitos quantitativos, executa cinco consultas com `g.triples()` e operações SPARQL de consulta e atualização.

> Uso acadêmico: as recomendações cadastradas são exemplos de modelagem e não substituem profissionais de saúde.

In [ ]:
# No Google Colab, descomente se necessário:
# !pip install -q rdflib

from pathlib import Path
from rdflib import Graph, Namespace, RDF, RDFS, OWL
from rdflib.namespace import XSD

FIT = Namespace("http://example.org/fittech#")
TTL_PATH = Path("fittech.ttl")

if not TTL_PATH.exists():
    raise FileNotFoundError("Coloque fittech.ttl na mesma pasta do notebook.")

g = Graph()
g.parse(TTL_PATH, format="turtle")
g.bind("fit", FIT)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("owl", OWL)

print(f"Base carregada com sucesso: {len(g)} triplas.")

## 1. Validação automática dos requisitos da base

In [ ]:
classes = set(g.subjects(RDF.type, OWL.Class))
obj_props = set(g.subjects(RDF.type, OWL.ObjectProperty))
data_props = set(g.subjects(RDF.type, OWL.DatatypeProperty))

schema_types = {OWL.Class, OWL.ObjectProperty, OWL.DatatypeProperty, OWL.Ontology,
                OWL.FunctionalProperty, OWL.InverseFunctionalProperty,
                OWL.SymmetricProperty, OWL.TransitiveProperty}
individuals = {
    s for s, _, o in g.triples((None, RDF.type, None))
    if o not in schema_types and o not in {RDF.Property}
}

owl_constructs = {
    "inverseOf": len(list(g.triples((None, OWL.inverseOf, None)))),
    "FunctionalProperty": len(set(g.subjects(RDF.type, OWL.FunctionalProperty))),
    "InverseFunctionalProperty": len(set(g.subjects(RDF.type, OWL.InverseFunctionalProperty))),
    "SymmetricProperty": len(set(g.subjects(RDF.type, OWL.SymmetricProperty))),
    "TransitiveProperty": len(set(g.subjects(RDF.type, OWL.TransitiveProperty))),
    "disjointWith": len(list(g.triples((None, OWL.disjointWith, None)))),
}

print(f"Classes OWL: {len(classes)}")
print(f"Propriedades de objeto: {len(obj_props)}")
print(f"Propriedades de dados: {len(data_props)}")
print(f"Indivíduos tipados: {len(individuals)}")
print(f"Total de triplas: {len(g)}")
print("Construções OWL:", owl_constructs)

assert len(classes) >= 8
assert len(obj_props) >= 5
assert len(data_props) >= 5
assert len(individuals) >= 25
assert len(g) >= 50
assert sum(v > 0 for v in owl_constructs.values()) >= 5
print("\n✅ Todos os requisitos quantitativos mínimos foram atingidos.")

## 2. Consultas com `g.triples()`

In [ ]:
def mostrar_triplas(numero, proposito, padrao, limite=20):
    print("=" * 80)
    print(f"Consulta g.triples() {numero}: {proposito}")
    print(f"Padrão: {padrao}")
    resultados = list(g.triples(padrao))
    for s, p, o in resultados[:limite]:
        print(f"- {s.n3(g.namespace_manager)} | {p.n3(g.namespace_manager)} | {o.n3(g.namespace_manager)}")
    if len(resultados) > limite:
        print(f"... mais {len(resultados) - limite} resultado(s).")
    print(f"Total: {len(resultados)}\n")
    return resultados

# 1) sujeito fixo; predicado e objeto livres
mostrar_triplas(1, "Todos os dados cadastrados sobre Ana", (FIT.ana, None, None))

# 2) predicado fixo; sujeito e objeto livres
mostrar_triplas(2, "Todos os usuários e seus objetivos", (None, FIT.temObjetivo, None))

# 3) objeto fixo; sujeito e predicado livres
mostrar_triplas(3, "Relações que apontam para Creatina", (None, None, FIT.creatina))

# 4) sujeito e predicado fixos
mostrar_triplas(4, "Suplementos utilizados por Bruno", (FIT.bruno, FIT.usaSuplemento, None))

# 5) predicado e objeto fixos
mostrar_triplas(5, "Indivíduos classificados como Usuário", (None, RDF.type, FIT.Usuario))

## 3. Funções auxiliares para SPARQL

In [ ]:
PREFIXOS = """
PREFIX fit: <http://example.org/fittech#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
"""

def executar_select(numero, proposito, consulta):
    print("=" * 80)
    print(f"SPARQL {numero}: {proposito}")
    resultado = g.query(PREFIXOS + consulta)
    print(" | ".join(str(v) for v in resultado.vars))
    linhas = list(resultado)
    for linha in linhas:
        print(" | ".join(str(valor) for valor in linha))
    print(f"Total: {len(linhas)}\n")
    return linhas

### SPARQL 1 — SELECT básico

In [ ]:
executar_select(1, "Listar usuários e seus objetivos", """
SELECT ?nome ?objetivo
WHERE {
    ?usuario a fit:Usuario ;
             fit:nome ?nome ;
             fit:temObjetivo ?obj .
    ?obj rdfs:label ?objetivo .
}
""")

### SPARQL 2 — SELECT com FILTER

In [ ]:
executar_select(2, "Listar usuários com 30 anos ou mais", """
SELECT ?nome ?idade
WHERE {
    ?usuario a fit:Usuario ; fit:nome ?nome ; fit:idade ?idade .
    FILTER(?idade >= 30)
}
ORDER BY ?idade
""")

### SPARQL 3 — SELECT com ORDER BY

In [ ]:
executar_select(3, "Ordenar usuários do maior para o menor peso", """
SELECT ?nome ?peso
WHERE {
    ?usuario a fit:Usuario ; fit:nome ?nome ; fit:pesoKg ?peso .
}
ORDER BY DESC(?peso)
""")

### SPARQL 4 — SELECT com agregação

In [ ]:
executar_select(4, "Contar usuários por tipo de objetivo", """
SELECT ?objetivo (COUNT(?usuario) AS ?quantidade)
WHERE {
    ?usuario a fit:Usuario ; fit:temObjetivo ?obj .
    ?obj rdfs:label ?objetivo .
}
GROUP BY ?objetivo
ORDER BY DESC(?quantidade)
""")

### SPARQL 5 — ASK

In [ ]:
print("=" * 80)
print("SPARQL 5: Existe usuário intolerante à lactose que busca hipertrofia?")
resultado_ask = g.query(PREFIXOS + """
ASK WHERE {
    ?usuario a fit:Usuario ;
             fit:restricaoAlimentar "intolerancia_lactose" ;
             fit:temObjetivo fit:objetivoHipertrofia .
}
""")
print("Resposta:", bool(resultado_ask), "\n")

### SPARQL 6 — CONSTRUCT

In [ ]:
print("=" * 80)
print("SPARQL 6: Construir um subgrafo de recomendações resumidas")
subgrafo = g.query(PREFIXOS + """
CONSTRUCT {
    ?usuario fit:recomendacaoResumida ?planoTreino .
}
WHERE {
    ?usuario a fit:Usuario ; fit:seguePlanoTreino ?planoTreino .
}
""")
print(subgrafo.serialize(format="turtle"))
print(f"Total no subgrafo: {len(subgrafo)} triplas.\n")

## 4. SPARQL Update

Os updates são feitos apenas na memória. O arquivo `fittech.ttl` original não é alterado.

### SPARQL 7 — INSERT DATA

In [ ]:
print("=" * 80)
print("SPARQL 7: Inserir temporariamente o suplemento Beta-alanina")
g.update(PREFIXOS + """
INSERT DATA {
    fit:betaAlanina a fit:Suplemento ;
        rdfs:label "Beta-alanina"@pt ;
        fit:doseDiaria "2 a 4 g" .
    fit:ana fit:usaSuplemento fit:betaAlanina .
}
""")
print("INSERT executado.\n")

### SPARQL 8 — Verificação do INSERT

In [ ]:
executar_select(8, "Verificar os suplementos de Ana após o INSERT", """
SELECT ?suplemento
WHERE {
    fit:ana fit:usaSuplemento ?s .
    ?s rdfs:label ?suplemento .
}
ORDER BY ?suplemento
""")

### SPARQL 9 — DELETE DATA

In [ ]:
print("=" * 80)
print("SPARQL 9: Remover a associação temporária de Ana com Beta-alanina")
g.update(PREFIXOS + """
DELETE DATA {
    fit:ana fit:usaSuplemento fit:betaAlanina .
}
""")
print("DELETE executado.\n")

### SPARQL 10 — DELETE/INSERT WHERE

In [ ]:
print("=" * 80)
print("SPARQL 10: Atualizar a frequência do treino ABC de 3 para 4 dias")
g.update(PREFIXOS + """
DELETE { fit:treinoABC fit:frequenciaSemanal ?frequenciaAntiga . }
INSERT { fit:treinoABC fit:frequenciaSemanal 4 . }
WHERE  { fit:treinoABC fit:frequenciaSemanal ?frequenciaAntiga . }
""")
print("DELETE/INSERT executado.\n")

### SPARQL 11 — Verificação final dos updates

In [ ]:
executar_select(11, "Confirmar a nova frequência do treino ABC", """
SELECT ?frequencia
WHERE {
    fit:treinoABC fit:frequenciaSemanal ?frequencia .
}
""")

associacao_existe = bool(g.query(PREFIXOS + """
ASK WHERE { fit:ana fit:usaSuplemento fit:betaAlanina . }
"""))
print("Ana ainda usa Beta-alanina?", associacao_existe)
print("Observação: o indivíduo Beta-alanina continua no grafo, mas a associação com Ana foi removida.")
print("O arquivo Turtle original continua intacto, pois os updates ocorreram apenas em memória.")

## 5. Resumo final

In [ ]:
print("Mini-Projeto 3 executado com sucesso.")
print(f"Triplas no grafo em memória após os updates: {len(g)}")
print("Consulte o README.md para instruções de execução e descrição da taxonomia.")